# Lab 6 — Serving on a TPU: what changes when there are no kernels

**The claim you should be able to make when you finish:** *"I have served a
model on a TPU. The thing that surprised me was that shapes are part of the
program — every new sequence length is a recompile, so serving is built around
padding buckets rather than around kernels."*

Colab is the cheapest way in the world to touch a TPU: change the runtime type
and you have one. That is the entire reason this lab is here — not because you
will deploy on TPUs next week, but because seeing a second execution model makes
the first one legible. Most of what feels like "how inference works" is actually
"how CUDA works".

**Runtime → Change runtime type → TPU v2-8 (or v5e) → Save.**

The notebook still runs without one — JAX will use CPU and every lesson except
the absolute numbers survives.

### The three differences that matter

1. **You do not write kernels.** You write array code and XLA fuses it. There is
   no `nvcc`, no occupancy tuning; there is also no dropping to CUDA when the
   compiler makes a bad choice. (Pallas exists for that, and is a different lab.)
2. **Shapes are compile-time constants.** A new sequence length is a new
   program. This is *the* structural difference for serving.
3. **Sharding is declarative.** You describe how arrays map onto a device mesh
   and the compiler inserts the collectives. There is no tensor-parallel
   codepath to write, only an annotation to get right.

In [ ]:
# Cell 1 — bootstrap. JAX is preinstalled on Colab TPU runtimes; installing the
# wrong wheel is the classic way to lose an hour, so we check before touching it.
REPO   = "https://github.com/lsgrep/serv.git"
BRANCH = "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

import jax, jax.numpy as jnp
print("jax", jax.__version__)
print("backend:", jax.default_backend())
for i, d in enumerate(jax.devices()):
    print(f"  device {i}: {d.device_kind}  {d}")

ON_TPU = jax.default_backend() == "tpu"
if not ON_TPU:
    print("\nNo TPU attached — Runtime > Change runtime type > TPU.")
    print("Everything below still runs on CPU; only the timings change.")

## 1. A decoder, in about sixty lines of array code

No framework, no checkpoint: random weights at GPT-2-small shapes. The point is
the *execution model*, and weights would only slow the download.

Note what is absent: no kernel, no thread block, no shared memory. Attention is
a couple of `einsum`s and XLA decides how to run them.

In [ ]:
import functools
import jax, jax.numpy as jnp
import numpy as np

# GPT-2 small shapes.
L, H, D, V = 12, 12, 64, 50257
HID = H * D
FFN = 4 * HID

def init_params(key, dtype=jnp.bfloat16):
    ks = jax.random.split(key, 8)
    scale = 0.02
    def rnd(k, *shape):
        return (jax.random.normal(k, shape) * scale).astype(dtype)
    return {
        "embed": rnd(ks[0], V, HID),
        "pos":   rnd(ks[1], 1024, HID),
        "wq":    rnd(ks[2], L, HID, HID),
        "wk":    rnd(ks[3], L, HID, HID),
        "wv":    rnd(ks[4], L, HID, HID),
        "wo":    rnd(ks[5], L, HID, HID),
        "w1":    rnd(ks[6], L, HID, FFN),
        "w2":    rnd(ks[7], L, FFN, HID),
    }

def layer_norm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / jnp.sqrt(var + eps)

def decode_step(params, cache, token, pos):
    """One token, for a batch. `cache` is (k, v), each [L, B, T, H, D].

    The cache is a *fixed-size preallocated array* updated with a dynamic slice
    — not a list that grows. On an accelerator with static shapes there is no
    other option, and this constraint is the whole flavour of TPU serving.
    """
    k_cache, v_cache = cache
    x = params["embed"][token] + params["pos"][pos]          # [B, HID]

    for i in range(L):
        h = layer_norm(x)
        q = h @ params["wq"][i]
        k = h @ params["wk"][i]
        v = h @ params["wv"][i]

        B = q.shape[0]
        q = q.reshape(B, H, D)
        k_cache = k_cache.at[i, :, pos].set(k.reshape(B, H, D))
        v_cache = v_cache.at[i, :, pos].set(v.reshape(B, H, D))

        # Attend over the whole preallocated window and mask the future. Doing
        # the full-width matmul every step is wasteful on paper and often free
        # in practice, because the shape must be static anyway.
        scores = jnp.einsum("bhd,bthd->bht", q, k_cache[i]) / jnp.sqrt(D).astype(q.dtype)
        mask = (jnp.arange(k_cache.shape[2]) <= pos)[None, None, :]
        scores = jnp.where(mask, scores, -1e30)
        probs = jax.nn.softmax(scores, axis=-1)
        attn = jnp.einsum("bht,bthd->bhd", probs, v_cache[i]).reshape(B, HID)
        x = x + attn @ params["wo"][i]

        h = layer_norm(x)
        x = x + jax.nn.gelu(h @ params["w1"][i]) @ params["w2"][i]

    logits = layer_norm(x) @ params["embed"].T
    return logits, (k_cache, v_cache)

print(f"model: {L} layers, {H} heads x {D} dim, hidden {HID}, vocab {V}")

In [ ]:
import time

BATCH, MAX_LEN = 8, 512
DTYPE = jnp.bfloat16 if ON_TPU else jnp.float32   # TPUs are bf16-native

key = jax.random.PRNGKey(0)
params = init_params(key, DTYPE)

def empty_cache(batch, max_len, dtype):
    shape = (L, batch, max_len, H, D)
    return (jnp.zeros(shape, dtype), jnp.zeros(shape, dtype))

n_params = sum(np.prod(v.shape) for v in params.values())
kv_bytes = 2 * L * BATCH * MAX_LEN * H * D * jnp.dtype(DTYPE).itemsize
print(f"parameters: {n_params/1e6:.0f}M ({n_params * jnp.dtype(DTYPE).itemsize / 1024**2:.0f} MiB)")
print(f"KV cache for batch {BATCH} x {MAX_LEN} tokens: {kv_bytes/1024**2:.0f} MiB")
print(f"  -> {2 * L * H * D * jnp.dtype(DTYPE).itemsize / 1024:.0f} KiB per token, "
      "which is the same formula as lab 3")

## 2. Compilation is a first-class cost

`jax.jit` traces your function into a graph and XLA compiles it for the exact
shapes it saw. The first call pays for compilation; later calls with the *same
shapes* are free.

`donate_argnums` matters more than it looks: it tells XLA the cache buffer can
be overwritten in place. Without it, every decode step allocates a fresh copy of
the entire KV cache — which is a real bug people ship, and it turns a
bandwidth-bound step into a bandwidth-catastrophe step.

In [ ]:
step = jax.jit(decode_step, donate_argnums=(1,))

cache = empty_cache(BATCH, MAX_LEN, DTYPE)
token = jnp.zeros((BATCH,), jnp.int32)

t0 = time.perf_counter()
logits, cache = step(params, cache, token, 0)
logits.block_until_ready()
compile_s = time.perf_counter() - t0

t0 = time.perf_counter()
for pos in range(1, 21):
    logits, cache = step(params, cache, token, pos)
logits.block_until_ready()
run_s = (time.perf_counter() - t0) / 20

print(f"first call (compile + run): {compile_s*1000:>9,.1f} ms")
print(f"steady state per step:      {run_s*1000:>9,.1f} ms")
print(f"compile was worth {compile_s/run_s:,.0f} steps")
print(f"\nthroughput: {BATCH/run_s:,.0f} tokens/s at batch {BATCH}")

In [ ]:
# Without donation, the cache is copied every step. Same maths, worse serving.
step_nodonate = jax.jit(decode_step)

cache2 = empty_cache(BATCH, MAX_LEN, DTYPE)
logits, cache2 = step_nodonate(params, cache2, token, 0)
logits.block_until_ready()

t0 = time.perf_counter()
for pos in range(1, 21):
    logits, cache2 = step_nodonate(params, cache2, token, pos)
logits.block_until_ready()
nodonate_s = (time.perf_counter() - t0) / 20

print(f"donated buffer:     {run_s*1000:>8.2f} ms/step")
print(f"not donated:        {nodonate_s*1000:>8.2f} ms/step")
print(f"cost of the copy:   {nodonate_s/run_s:>8.2f}x")
print("\nThe cache is the biggest array in the program. Copying it once per token")
print("is the accelerator equivalent of an accidental O(n^2).")

## 3. The lesson that has no CUDA analogue: shape changes are recompiles

Change the batch size, the cache width, or any other shape, and XLA compiles a
new program. On a GPU server this costs nothing — kernels take shapes at
runtime. On a TPU it means an unlucky request can block the server for seconds.

Hence the shape of every TPU serving stack:

* **pad to buckets** — a handful of sequence lengths (512, 1024, 2048...) rather
  than exact lengths,
* **fixed batch sizes** — pad the batch too, and eat the waste,
* **warm up every bucket at startup**, before traffic arrives,
* **treat a cache-miss compile as an incident**, because to a user it looks like
  one.

Measure it directly.

In [ ]:
import matplotlib.pyplot as plt
from servlab.plots import use_style, SERIES, STATUS

use_style()
buckets = [128, 256, 512, 1024]
compile_times, step_times = [], []

for width in buckets:
    c = empty_cache(BATCH, width, DTYPE)
    t0 = time.perf_counter()
    out, c = step(params, c, token, 0)          # new shape -> new compile
    out.block_until_ready()
    compile_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    for pos in range(1, 11):
        out, c = step(params, c, token, pos)
    out.block_until_ready()
    step_times.append((time.perf_counter() - t0) / 10)
    print(f"cache width {width:>5}: compile {compile_times[-1]*1000:>8,.0f} ms   "
          f"step {step_times[-1]*1000:>6.2f} ms")

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar([str(b) for b in buckets], [c * 1000 for c in compile_times], color=SERIES[0], width=0.55)
ax.set_ylabel("first-call latency (ms)")
ax.set_xlabel("KV cache width (padding bucket)")
ax.set_title("every bucket is a separate compile — warm them all at startup")
for i, c in enumerate(compile_times):
    ax.annotate(f"{c*1000:,.0f} ms", xy=(i, c * 1000), xytext=(0, 3),
                textcoords="offset points", ha="center", fontsize=9, color="#52514e")
ax.margins(y=0.18)
plt.show()

print("\nA request that lands on a cold bucket waits for the whole bar.")
print("This is why TPU serving pads aggressively: fewer programs, all warm.")

In [ ]:
# The cost of padding, since you now pay for the whole window every step.
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(buckets, [s * 1000 for s in step_times], marker="o", color=SERIES[0])
ax.set_xscale("log", base=2)
ax.set_xlabel("cache width (padded)"); ax.set_ylabel("ms per decode step")
ax.set_title("padding is not free: a wider window is a bigger attention matmul")
plt.show()

print("The trade: fewer buckets means fewer compiles and more wasted compute.")
print("Real stacks land on 3-5 buckets. That number is an engineering decision")
print("you should be able to defend from a latency histogram of your traffic.")

## 4. Sharding, declared rather than implemented

Tensor parallelism on a GPU means writing (or adopting) a codepath that splits
matmuls and inserts all-reduces. In JAX you build a device mesh, say which axis
each array is split along, and XLA inserts the collectives.

A Colab TPU gives you 8 cores, which is enough to see it work.

In [ ]:
import jax
from jax.sharding import Mesh, PartitionSpec as P, NamedSharding

devs = jax.devices()
print(f"{len(devs)} devices")

if len(devs) >= 2:
    import numpy as np
    mesh = Mesh(np.array(devs).reshape(len(devs)), ("model",))
    # Split the FFN's hidden dimension across devices — the classic
    # tensor-parallel split, expressed as one annotation per array.
    w1 = jax.device_put(params["w1"], NamedSharding(mesh, P(None, None, "model")))
    w2 = jax.device_put(params["w2"], NamedSharding(mesh, P(None, "model", None)))
    print("w1 sharding:", w1.sharding)
    print("w2 sharding:", w2.sharding)
    print("\nXLA now inserts the reduce after the second matmul. No codepath was")
    print("written for it — the annotation is the implementation.")
else:
    print("single device — attach a TPU runtime to see sharding do anything")

## 5. Put the numbers next to lab 3

Same formula, different silicon. The per-token KV size is a property of the
model, not the hardware, so it transfers exactly. What changes is bandwidth,
memory per chip, and the compile-time constraint.

In [ ]:
from servlab import napkin as nk

spec = nk.MODELS["gpt2"]
print(f"GPT-2 KV per token: {nk.human_bytes(nk.kv_bytes_per_token(spec))} (fp16)")
print(f"measured here:      {2 * L * H * D * jnp.dtype(DTYPE).itemsize / 1024:.0f} KiB "
      f"({DTYPE.__name__})")
print()
print(f"this run:  batch {BATCH}, {run_s*1000:.2f} ms/step -> {BATCH/run_s:,.0f} tok/s")
print(f"T4 (predicted, same model, same batch): "
      f"{nk.decode_tokens_per_s('T4', spec, batch=BATCH, ctx_len=MAX_LEN):,.0f} tok/s")
print()
print("Do not read too much into the ratio: this is unoptimised array code with")
print("no flash-attention kernel and no fused layers, so it measures the")
print("execution model, not the hardware ceiling. The comparison worth making is")
print("structural, not numeric.")

## What to be able to say afterwards

1. **Shapes are part of the program.** Serving on a TPU means padding buckets
   and warmed-up compilations, and a cold bucket is a user-visible stall.
2. **Buffer donation is not an optimisation, it is a correctness-of-design
   issue** for anything holding a KV cache.
3. **Sharding is declarative** — you annotate, the compiler implements. That is
   genuinely less code than a GPU tensor-parallel path, and genuinely less
   escape hatch when the compiler is wrong.
4. **The KV formula does not care about the vendor.** Memory arithmetic
   transfers; kernels do not.
5. **Why anyone bothers:** TPU pods have very high interconnect bandwidth per
   dollar, which matters for large-model serving where the model spans chips.
   For a 3B model on one card it is not the interesting question.

### The end of the ladder

That is the last notebook. Two things worth doing now:

* **Go back to `terminal/`** and redo labs 1-2 on a rented box — tmux, curl,
  logs. The notebooks built the intuition; the terminal rehearses the
  performance, and an interview is a performance.
* **Write down, per lab, the one-sentence claim** at the top of each notebook and
  check you can defend it without opening the notebook. If you can, the ladder
  did its job.